# RAASTA — Download datasets to Google Drive (one by one)\n\n**Rule:** Data stays on Drive. Laptop only gets final `.pt` / `.tflite` later.\n\n1. Runtime → **T4 GPU** optional for download (CPU is fine)\n2. Run **Setup** cells\n3. Run **only one dataset cell** at a time\n4. Tell your teammate/agent when a download finishes before the next one\n\nFolder layout:\n```\nMyDrive/raasta/\n  raw/       ← zips (temporary)\n  extracted/ ← unzipped sources\n  merged/    ← YOLO format (later)\n  weights/   ← models (later)\n```

## 0) Setup — mount Drive + folders

In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, zipfile, subprocess, sys

drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/raasta')
RAW = ROOT / 'raw'
EXTRACTED = ROOT / 'extracted'
MERGED = ROOT / 'merged'
WEIGHTS = ROOT / 'weights'

for p in (RAW, EXTRACTED, MERGED, WEIGHTS):
    p.mkdir(parents=True, exist_ok=True)

print('Drive folder:', ROOT)
print('OK — folders ready')

In [ ]:
# Free space check (Drive mount stats — approximate)
!df -h /content/drive | tail -n 1
print('\nAlso check: https://drive.google.com/drive/quota')
print('You have ~12 GB free — download ONE dataset at a time.')

In [ ]:
def download(url: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 1_000_000:
        print(f'Already exists ({dest.stat().st_size/1e9:.2f} GB): {dest}')
        return dest
    print('Downloading →', dest)
    # wget is resilient for large files
    cmd = ['wget', '-c', '-O', str(dest), url]
    subprocess.check_call(cmd)
    print(f'Done: {dest.stat().st_size/1e9:.2f} GB')
    return dest

def unzip_to(zip_path: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    marker = out_dir / '_EXTRACT_OK'
    if marker.exists():
        print('Already extracted:', out_dir)
        return out_dir
    print('Extracting', zip_path.name, '→', out_dir)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(out_dir)
    marker.write_text('ok')
    print('Extract done')
    return out_dir

print('Helpers ready')

## 1) Dataset #1 — RDD2022 India (~0.5 GB)  ← RUN THIS FIRST\n\nSource: https://github.com/sekilab/RoadDamageDetector  \nClasses later: cracks + potholes

In [ ]:
INDIA_URL = (
    'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/'
    '2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_India.zip'
)
india_zip = RAW / 'RDD2022_India.zip'
download(INDIA_URL, india_zip)
unzip_to(india_zip, EXTRACTED / 'RDD2022_India')

# Quick peek
!find /content/drive/MyDrive/raasta/extracted/RDD2022_India -maxdepth 3 -type d | head -n 40
print('\n✅ Dataset #1 ready. Reply in chat: "India done" before next download.')

## 2) Dataset #2 — RDD2022 Japan (~1 GB)  ← only after India done

In [ ]:
# Run ONLY after India finished and Drive still has free space
JAPAN_URL = (
    'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/'
    '2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_Japan.zip'
)
japan_zip = RAW / 'RDD2022_Japan.zip'
download(JAPAN_URL, japan_zip)
unzip_to(japan_zip, EXTRACTED / 'RDD2022_Japan')
print('\n✅ Dataset #2 ready. Reply: "Japan done"')

## Optional — delete a raw zip after extract (frees Drive)\nUncomment and run only when extract folder looks correct.

In [ ]:
# Example: free ~0.5 GB after India extract is verified
# (RAW / 'RDD2022_India.zip').unlink(missing_ok=True)
# print('Deleted India zip')
print('Left commented on purpose — delete only after we confirm extract.')

## Later datasets (do not run yet)\nWe will add cells for: SBP-YOLO, Kaggle road_anomaly, BDD100K person filter, COCO subset, ExDark, Open Images filtered.\n\n**No local phone recording** — public internet datasets only (as agreed).